# Lab: ice-stream response and volume above flotation

The prognostic and climate-forcing chapters assembled the pieces of a forecast, a thickness-evolution equation, a flow model to supply the flux, and a parameterized climate to force it. This lab puts them together. We build a synthetic marine-terminating ice stream in icepack, spin it up to a steady state, perturb its climate, and track the response in the one number that connects ice dynamics to the coastline, the **volume above flotation** (VAF), the ice that would raise sea level if it were lost. Running the same experiment on two beds, one sloping steadily seaward and one with an overdeepening, previews the asymmetry that {doc}`../climate/glacier-variations` explains with two ordinary differential equations.

This notebook is not executed when the book is built. Run it in the Docker container described in {doc}`running-icepack-docker`; the spin-up loops take minutes to tens of minutes at the default resolution, and the text notes where to coarsen if you want faster turnaround.

## Volume above flotation

Ice already displacing its weight in seawater does not change sea level when it melts. The glaciologically active volume is the part of the column above flotation. With bed elevation $b$ (negative below sea level) and densities $\rho_i$ and $\rho_w$, the thickness above flotation is

$$
h_{af} = h - \frac{\rho_w}{\rho_i}\,\max(0, -b),
$$

and the volume above flotation is the integral of $\max(h_{af}, 0)$ over the domain. Losing $362.5$ Gt of VAF raises global mean sea level by one millimeter, the bookkeeping used in {doc}`../climate/sea-level`. Everything this lab measures is a time series of this integral.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import firedrake
from firedrake import (
    Constant, interpolate, assemble, conditional, max_value, sqrt, inner, dx
)
import icepack
import icepack.models, icepack.solvers

# verify against your icepack version: constant names and locations
from icepack.constants import (
    ice_density as rho_I,
    water_density as rho_W,
    gravity as g,
)
print(f"rho_I = {rho_I}, rho_W = {rho_W}, g = {g}")

## Task 1: geometry, bed, and a steady state

The domain is a rectangle 640 km long and 80 km wide, flowing in $x$. We define two beds, uniform across-flow. Bed A slopes steadily down toward the ocean (prograde everywhere). Bed B is the overdeepened profile used by the marine ice-sheet intercomparison exercises, prograde near the divide and near the coast but retrograde in between, where the bed deepens inland.

Resolution note: $64\times 8$ cells with piecewise-linear elements is deliberately coarse, enough to expose the dynamics this lab is after. Grounding-line problems converge slowly with resolution, and a research-grade experiment would refine the grounding zone by an order of magnitude; see the discussion of resolution in {doc}`../modeling/prognostic-problem`.

In [ ]:
Lx, Ly = 640e3, 80e3
nx, ny = 64, 8   # coarsen to 48 x 6 for quick experiments

mesh = firedrake.RectangleMesh(nx, ny, Lx, Ly)
Q = firedrake.FunctionSpace(mesh, "CG", 1)
V = firedrake.VectorFunctionSpace(mesh, "CG", 1)

x, y = firedrake.SpatialCoordinate(mesh)

# Bed A: prograde, deepening steadily toward the ocean
bed_A = interpolate(150 - 1.0e-3 * x, Q)

# Bed B: overdeepened (MISMIP-style polynomial in x/750 km)
xi = x / 750e3
bed_B = interpolate(729 - 2184.8 * xi**2 + 1031.72 * xi**4 - 151.72 * xi**6, Q)

fig, axes = plt.subplots()
xs = np.linspace(0, Lx, 257)
for b, label in ((bed_A, "Bed A (prograde)"), (bed_B, "Bed B (overdeepened)")):
    axes.plot(xs / 1e3, [b.at(xx, Ly / 2) for xx in xs], label=label)
axes.axhline(0, color="k", lw=0.5)
axes.set_xlabel("x (km)"); axes.set_ylabel("bed elevation (m)")
axes.legend();

In [ ]:
# Initial thickness, surface, and physical fields
h0 = interpolate(Constant(1500.0), Q)
h = h0.copy(deepcopy=True)

bed = bed_A   # switch to bed_B in Task 3
s = icepack.compute_surface(thickness=h, bed=bed)

T = Constant(255.0)
A = icepack.rate_factor(T)   # verify: rate_factor location in your icepack version

C = Constant(0.05)           # Weertman friction coefficient; tune with sliding-laws lab in mind
a_smb = interpolate(Constant(0.3), Q)   # accumulation, m/yr ice equivalent

The friction law needs one modification from the form used in the sliding-laws lab. Near the grounding line the water pressure under the ice approaches the ice overburden, and the basal traction must fall to zero where the ice goes afloat. Following the pattern of icepack's marine ice-sheet demonstrations, we scale the Weertman friction by the ratio of effective pressure to overburden, computed from the depth of the bed below sea level.

In [ ]:
import icepack.models.friction

def ramped_weertman(**kwargs):
    # verify against your icepack version: friction signature and bed_friction name
    u = kwargs["velocity"]
    h = kwargs["thickness"]
    s = kwargs["surface"]
    C = kwargs["friction"]

    p_W = rho_W * g * max_value(0, h - s)   # water pressure from flotation depth
    p_I = rho_I * g * h
    ramp = max_value(0, 1 - p_W / p_I)      # 1 grounded, 0 at flotation
    return icepack.models.friction.bed_friction(velocity=u, friction=C * ramp)

model = icepack.models.IceStream(friction=ramped_weertman)
solver = icepack.solvers.FlowSolver(model, dirichlet_ids=[1], side_wall_ids=[3, 4])

u0 = interpolate(firedrake.as_vector((100 * x / Lx, 0)), V)
u = solver.diagnostic_solve(
    velocity=u0, thickness=h, surface=s, fluidity=A, friction=C,
)

In [ ]:
lam = rho_W / rho_I

def volume_above_flotation(h, bed):
    h_af = h - lam * max_value(0, -bed)
    return assemble(max_value(h_af, 0) * dx)

def sle_mm(dvaf_m3):
    """Sea-level equivalent (mm) of a VAF change in cubic meters of ice."""
    return rho_I * dvaf_m3 / 3.625e14   # 362.5 Gt per mm

def grounding_line_x(h, bed, n=513):
    """Centerline grounding-line position from the sign change of h_af."""
    xs = np.linspace(0, Lx, n)
    haf = np.array([
        h.at(xx, Ly / 2) - lam * max(0.0, -bed.at(xx, Ly / 2)) for xx in xs
    ])
    grounded = np.where(haf > 0)[0]
    return xs[grounded[-1]] if len(grounded) else 0.0

print(f"initial VAF = {volume_above_flotation(h, bed):.3e} m^3")

In [ ]:
# Spin-up to steady state on Bed A
dt = 2.0
num_steps = 1500          # 3000 years; lengthen if VAF is still drifting

vaf_series = []
for step in range(num_steps):
    h = solver.prognostic_solve(
        dt, thickness=h, velocity=u, accumulation=a_smb,
    )
    h.interpolate(max_value(h, 10.0))   # minimum thickness over the ocean
    s = icepack.compute_surface(thickness=h, bed=bed)
    u = solver.diagnostic_solve(
        velocity=u, thickness=h, surface=s, fluidity=A, friction=C,
    )
    vaf_series.append(volume_above_flotation(h, bed))

vaf_series = np.array(vaf_series)
t = dt * np.arange(1, num_steps + 1)

fig, axes = plt.subplots()
axes.plot(t, vaf_series / 1e12)
axes.set_xlabel("time (yr)"); axes.set_ylabel("VAF (10$^3$ km$^3$)")
axes.set_title("Spin-up on Bed A");

Confirm before continuing that the VAF curve has flattened; the relative drift over the last 500 years should be well under a percent. Save the spun-up state, since both forcing experiments start from it.

## Task 2: a step in the climate and the relaxation it produces

Force the steady stream two ways, separately: an atmospheric perturbation, stepping the accumulation down by 30 percent, and an oceanic one, applying a melt rate to the floating portion of the domain only. The floating mask is the same flotation calculation the VAF integral uses, so ocean melt automatically follows the grounding line as it migrates.

In [ ]:
h_steady = h.copy(deepcopy=True)
u_steady = u.copy(deepcopy=True)
vaf_0 = volume_above_flotation(h_steady, bed)
gl_0 = grounding_line_x(h_steady, bed)

# Ocean forcing: melt where the ice is afloat
melt_rate = Constant(2.0)   # m/yr, applied to floating ice
h_af = h - lam * max_value(0, -bed)
forcing = interpolate(
    a_smb - conditional(h_af < 0, melt_rate, 0.0), Q,
)

num_steps = 500   # 1000 years of forced evolution
vaf_f, gl_f = [], []
for step in range(num_steps):
    h = solver.prognostic_solve(dt, thickness=h, velocity=u, accumulation=forcing)
    h.interpolate(max_value(h, 10.0))
    s = icepack.compute_surface(thickness=h, bed=bed)
    u = solver.diagnostic_solve(
        velocity=u, thickness=h, surface=s, fluidity=A, friction=C,
    )
    # refresh the floating mask so the melt follows the grounding line
    h_af = h - lam * max_value(0, -bed)
    forcing = interpolate(a_smb - conditional(h_af < 0, melt_rate, 0.0), Q)
    vaf_f.append(volume_above_flotation(h, bed))
    gl_f.append(grounding_line_x(h, bed))

vaf_f = np.array(vaf_f); gl_f = np.array(gl_f)
tf = dt * np.arange(1, num_steps + 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(tf, sle_mm(vaf_0 - vaf_f))
axes[0].set_xlabel("time (yr)"); axes[0].set_ylabel("sea-level contribution (mm)")
axes[1].plot(tf, (gl_f - gl_0) / 1e3)
axes[1].set_xlabel("time (yr)"); axes[1].set_ylabel("grounding-line change (km)");

In [ ]:
# Estimate the e-folding time of the VAF relaxation toward its new state
dvaf = vaf_f - vaf_f[-1]
mask = dvaf > 0.05 * dvaf[0]
p = np.polyfit(tf[mask], np.log(dvaf[mask]), 1)
print(f"e-folding time ~ {-1/p[0]:.0f} yr")

Three things to examine and write down. First, the shape of the VAF curve, a fast partial adjustment followed by a slower approach to the new balance; compare the fitted e-folding time against the interior estimate $\tau \sim H/\dot a$ and against the two-timescale structure derived in {doc}`../climate/glacier-variations`. Second, the grounding line, which retreats quickly at first and then settles at a new position on the prograde slope; the bed slope is what arrests it. Third, repeat the run with the accumulation step instead of the melt forcing and compare the response times; the two forcings enter the reduced model through different parameters ($P$ and $\Omega$), and the difference in pathway is measurable here.

## Task 3: the same forcing on the overdeepened bed

Rebuild the stream on Bed B, spin up exactly as in Task 1 (expect the steady grounding line to sit on the outer prograde slope), and apply the identical melt forcing. Track VAF and grounding-line position for at least 1500 years.

In [ ]:
# Rebuild on the overdeepened bed and rerun Tasks 1-2.
bed = bed_B
h = h0.copy(deepcopy=True)
s = icepack.compute_surface(thickness=h, bed=bed)
u = solver.diagnostic_solve(
    velocity=u_steady, thickness=h, surface=s, fluidity=A, friction=C,
)
# ... spin-up loop as in Task 1, then forcing loop as in Task 2 ...
# Plot the two sea-level-contribution curves (Bed A and Bed B) on shared axes.

## Task 4: synthesis

On Bed A the forced stream found a new steady state; on Bed B, once the grounding line is pushed onto the section where the bed deepens inland, it does not, and the VAF loss runs away until the grounding line reaches the next prograde reach. Answer in a few sentences each, with reference to {doc}`../climate/glacier-variations`:

1. Locate the steady-state grounding lines from your runs on the two bed profiles and compute the sign of the stability number $\Sigma$ at each. Does the sign predict what the simulations did?
2. The reduced model says the response gain scales as $1/\Sigma$ and the slow timescale diverges as $\Sigma \to 0$. Design (do not run) the cheapest icepack experiment that would test the divergence of the timescale.
3. Your melt forcing was applied uniformly to all floating ice. Real ocean melt concentrates near the grounding zone and acts partly by thinning the buttressing shelf, entering the reduced model through $\Omega$. Which features of your VAF curves would you expect to change under a more realistic melt pattern, and which are robust?
4. Convert your Bed B endpoint VAF loss to millimeters of sea level. What fraction of the initial VAF was committed once the grounding line entered the retrograde reach, and at what point in the time series could an observer first have known?